# Research Proposal: Modeling Daily Parking Ticket Counts in Milwaukee

### John Eckberg

## Overview
[Milwaukee plans 65,000 more parking tickets, longer meter hours to fill transportation budget gap](https://www.jsonline.com/story/news/local/milwaukee/2025/10/21/milwaukee-aims-for-550k-parking-tickets-longer-meter-hours-in-2026/86817755007/) The city of Milwaukee relies on parking ticket revenue in order to fill budget gaps, a pressing issue given the [current deficit](https://urbanmilwaukee.com/2025/03/12/mke-county-county-faces-47-million-2026-budget-deficit/) the county is facing.
This project plans to examine how daily parking ticket activity in Milwaukee varies across time and environmental conditions. Using parking citation records obtained through a Freedom of Information Act request (2014–2022), I propose to model daily ticket counts and evaluate whether temporal, weather, and holiday-related factors explain variation in enforcement volume.

## Research Question
What is the daily rate of parking tickets issued in Milwaukee, and which factors most strongly influence that rate (if any)? In particular, I propose to test whether day of week, month/seasonality, holidays, and precipitation are associated with higher or lower daily ticket counts, including by citation type.

## Hypothesis
- Null Hypothesis: day of week, month/seasonality, holidays, and precipitation do not affect daily ticket counts.
- Alternative Hypothesis: at least one temporal or environmental factor affects daily ticket counts.

## Data Sources
- **Parking citations:** Milwaukee parking ticket records, initially loaded for 2014, with fields including `ISSUENO`, `ISSUEDATE`, `LOCATIONDESC1`, and `VIODESCRIPTION`.
- **Holiday data:** U.S. holiday dates (2004–2021) pulled from Kaggle to identify atypical traffic/parking demand days.
- **Weather data:** Open-Meteo daily weather for Milwaukee, including maximum temperature, minimum temperature, precipitation, and snowfall.


## Method
The analysis would likely focus on a geographically meaningful subset of Milwaukee (e.g., downtown, Lower East Side, Deer District). Because the response variable is a daily count, I propose that the initial model choice would be a Poisson regression GLM.

### Currently Planned Features
- **Calendar effects:** day of week, month, weekend indicator, and potential seasonal terms.
- **Cyclical time encoding:** sine/cosine transforms for month and day-of-week to preserve circular structure.
- **Holiday effects:** binary `is_holiday` indicator, one-hot encoding for major holiday names.
- **Weather exposure:** `max_temp`, `min_temp`, `precipitation`, and `snowfall`; optional threshold features (e.g., heavy precipitation day).
- **Autocorrelation:** lagged ticket count features (e.g., 1-day and 7-day lags) and short rolling averages.



In [71]:
import pandas as pd
import requests
import kagglehub 
import openpyxl


In [72]:
# Loading in the parking data, just one year
tickets_df = pd.read_excel("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/tickets_csv/2014_parking_citations_issued_open_records_request.xlsx")

tickets_df.head(10)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION
0,474576465,2014-01-01,2444 PALMER ST,UNREGISTERED/ IMPROPERLY REGISTERED VEHICLE
1,474311040,2014-01-01,5419 GALENA ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
2,474311062,2014-01-01,2371 56TH ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
3,474353456,2014-01-01,1112 PLEASANT ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
4,474365522,2014-01-01,2969 46TH ST,PARKING PROHIBITED BY OFFICIAL SIGN
5,474365533,2014-01-01,3422 49TH ST,PARKED IN CROSSWALK
6,474311106,2014-01-01,2648 56TH ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
7,474417915,2014-01-01,2151 30TH ST,TOW-AWAY ZONE (BLOCKING DRIVEWAY)
8,474455380,2014-01-01,523 34TH ST,PARKING IN EXCESS OF 24 HOURS
9,474468400,2014-01-01,833 HAWLEY RD,PARKED POSTED PRIVATE PROPERTY


In [73]:
# load in holiday data (https://www.kaggle.com/datasets/donnetew/us-holiday-dates-2004-2021)

path = kagglehub.dataset_download("donnetew/us-holiday-dates-2004-2021")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\eckbergj\.cache\kagglehub\datasets\donnetew\us-holiday-dates-2004-2021\versions\1


In [74]:
holidays_df = pd.read_csv("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/US Holiday Dates (2004-2021).csv")

holidays_df.head(10)

,Date,Holiday,WeekDay,Month,Day,Year
0,2004-07-04,4th of July,Sunday,7,4,2004
1,2005-07-04,4th of July,Monday,7,4,2005
2,2006-07-04,4th of July,Tuesday,7,4,2006
3,2007-07-04,4th of July,Wednesday,7,4,2007
4,2008-07-04,4th of July,Friday,7,4,2008
5,2009-07-04,4th of July,Saturday,7,4,2009
6,2010-07-04,4th of July,Sunday,7,4,2010
7,2011-07-04,4th of July,Monday,7,4,2011
8,2012-07-04,4th of July,Wednesday,7,4,2012
9,2013-07-04,4th of July,Thursday,7,4,2013


In [75]:
# weather data (https://open-meteo.com/)
test_data = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=43.0389&longitude=-87.9065&start_date=2014-01-01&end_date=2014-12-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum&timezone=America%2FChicago")
test_data_json = test_data.json()

# minor cleaning of the weather data to get it into a dataframe
weather_df = pd.DataFrame(test_data_json['daily'])
weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df = weather_df.rename(columns={'time': 'date', 'temperature_2m_max': 'max_temp', 'temperature_2m_min': 'min_temp', 'precipitation_sum': 'precipitation','snowfall_sum': 'snowfall'})
weather_df.head(10)


,date,max_temp,min_temp,precipitation,snowfall
0,2014-01-01,-7.6,-13.3,1.3,1.33
1,2014-01-02,-8.2,-13.7,0.7,1.05
2,2014-01-03,-6.6,-17.2,0.0,0.00
3,2014-01-04,0.3,-6.1,0.7,0.91
4,2014-01-05,-4.3,-16.6,1.5,1.40
5,2014-01-06,-17.6,-25.0,0.0,0.21
6,2014-01-07,-15.9,-24.4,0.0,0.00
7,2014-01-08,-12.0,-17.4,0.0,0.00
8,2014-01-09,-3.9,-16.5,0.0,0.14
9,2014-01-10,3.1,-3.8,13.4,0.14


In [76]:
# clean and align date keys before joining
tickets_clean = tickets_df.copy()
holidays_clean = holidays_df.copy()
weather_clean = weather_df.copy()

tickets_clean['join_date'] = pd.to_datetime(tickets_clean['ISSUEDATE'], errors='coerce').dt.normalize()
holidays_clean['join_date'] = pd.to_datetime(holidays_clean['Date'], errors='coerce').dt.normalize()
weather_clean['join_date'] = pd.to_datetime(weather_clean['date'], errors='coerce').dt.normalize()

# merge tickets with holidays and weather on a consistent datetime key
combined_tickets_and_holidays_df = pd.merge(
    tickets_clean,
    holidays_clean,
    how='inner', # in practice should be a left join
    on='join_date'
 )
combined_all_df = pd.merge(
    combined_tickets_and_holidays_df,
    weather_clean,
    how='left',
    on='join_date',
    suffixes=('_holiday', '_weather')
 )

# combined_all_df = combined_all_df.drop(columns=['join_date','Date','date','ISSUEDATE'])
# combined_all_df.head(10)

# sample of the combined dataframe
combined_all_df.sample(5)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION,join_date,Date,Holiday,WeekDay,Month,Day,Year,date,max_temp,min_temp,precipitation,snowfall
12057,480505384,2014-11-11,4228 N 47TH ST,NIGHT PARKING,2014-11-11,2014-11-11,Veterans Day,Tuesday,11,11,2014,2014-11-11,10.7,-0.2,5.5,0.21
16703,481714984,2014-12-24,2528 E BELLEVIEW PL,METER PARKING VIOLATION,2014-12-24,2014-12-24,Christmas Eve,Wednesday,12,24,2014,2014-12-24,3.2,1.6,1.4,0.00
14054,481006643,2014-11-26,1132 N 6TH ST,METER PARKING VIOLATION,2014-11-26,2014-11-26,Thanksgiving Eve,Wednesday,11,26,2014,2014-11-26,-0.8,-6.8,0.0,0.00
14776,481731386,2014-12-24,1132 E EUCLID AVE,NIGHT PARKING - WRONG SIDE,2014-12-24,2014-12-24,Christmas Eve,Wednesday,12,24,2014,2014-12-24,3.2,1.6,1.4,0.00
7751,479389691,2014-08-30,3468 N DOWNER AVE,NIGHT PARKING,2014-08-30,2014-08-30,Labor Day Weekend,Saturday,9,30,2014,2014-08-30,24.7,21.3,0.4,0.00
